# Search Engine With Tools and Agents

In [8]:
1+1

2

In [27]:
# Arxiv - Research
# Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun # this is the tools which brings output when llm interact with it
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

# using langsmith for tracking

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_API_KEY"] = os.getenv('LANGCHAIN_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "search_engine_with_tool_and_agents"

In [10]:
# used the inbuilt tools of wikipedia

api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
# this is wrapper which allow as to scrap the wikipidia and bring summary of question and in this case it bring top 1 result and it content should less than 250 

# to run above wrapper we need WikipediaQueryRun
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

wiki.name

'wikipedia'

In [11]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [12]:
tools = [wiki,arxiv]

In [13]:
# Custome tools [Rag Tools]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"]=os.getenv('HF_TOKEN')

import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()


loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents = documents.split_documents(docs)
embedding = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2") 
vectorDb = FAISS.from_documents(documents,embedding)
retriever = vectorDb.as_retriever()
retriever


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7597.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020C3C2317B0>, search_kwargs={})

In [15]:
# make this retriever as tools
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith-search",
    "Search any information about langsmith"
)
retriever_tool.name

'langsmith-search'

In [16]:
tools=[arxiv,wiki,retriever_tool]

In [17]:
tools

[ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\MindsprintCourse\\GenAI_Apps\\Q&AChatbot\\search_engine_with_tools_and_agents\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000020C144D0C10>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000020C3C22A0E0>)]

In [ ]:
# [ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
#  WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\MindsprintCourse\\GenAI_Apps\\Q&AChatbot\\search_engine_with_tools_and_agents\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
#  StructuredTool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001A9D8826DD0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001A9FFE43760>)]

In [18]:
# now run all this tools with agents and llm models

# tools , llm-> AgentExecutor
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
api_key  = os.getenv('GROQ_API_KEY')
llm = ChatGroq(
        groq_api_key = api_key,
        model_name="openai/gpt-oss-20b",
        
    )
    

In [19]:
# prompt template 
from langchain_classic  import hub

prompt = hub.pull("hwchase17/openai-functions-agent")
print(prompt)
print(prompt.messages)


input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]

In [ ]:

# d:\MindsprintCourse\GenAI_Apps\Q&AChatbot\search_engine_with_tools_and_agents\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
#   from .autonotebook import tqdm as notebook_tqdm
# input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messages.function.FunctionMessageChunk, Tag(tag='FunctionMessageChunk')], typing.Annotated[langchain_core.messages.tool.ToolMessageChunk, Tag(tag='ToolMessageChunk')]], FieldInfo(annotation=NoneType, required=True, discriminator=Discriminator(discriminator=<function _get_type at 0x0000020C69FD0D30>, custom_error_type=None, custom_error_message=None, custom_error_context=None))]], 'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messages.function.FunctionMessageChunk, Tag(tag='FunctionMessageChunk')], typing.Annotated[langchain_core.messages.tool.ToolMessageChunk, Tag(tag='ToolMessageChunk')]], FieldInfo(annotation=NoneType, required=True, discriminator=Discriminator(discriminator=<function _get_type at 0x0000020C69FD0D30>, custom_error_type=None, custom_error_message=None, custom_error_context=None))]]} partial_variables={'chat_history': []} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'openai-functions-agent', 'lc_hub_commit_hash': 'a1655024b06afbd95d17449f21316291e0726f13dcfaf990cc0d18087ad689a5'} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]
# [SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]

In [20]:
# Agents
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [22]:
# agent executer
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [29]:
agent_executor.invoke({
    "input":"Tell me about Langsmith"
})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'langsmith'}`


LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place,

{'input': 'Tell me about Langsmith',
 'output': '**LangSmith** is a framework‑agnostic platform designed to simplify the end‑to‑end lifecycle of AI agents and large‑language‑model (LLM) applications. It brings together observability, evaluation, testing, and deployment into a single, cohesive workflow.\n\n| Feature | What it does | Why it matters |\n|---------|--------------|----------------|\n| **Observability** | Traces every request, response, and internal step your application takes. | Gives you full visibility into agent behavior, making debugging and performance tuning faster. |\n| **Evaluation** | Measure and track quality metrics (e.g., correctness, safety, bias) over time. | Ensures your models stay consistent, trustworthy, and aligned with business goals. |\n| **Prompt Engineering** | Version‑controlled prompt libraries, collaborative editing, and in‑app testing. | Lets teams iterate on prompts quickly and ship improvements without breaking existing functionality. |\n| **Test

In [30]:
agent_executor.invoke({
    "input":"what is difference between langchain and langgraph?"
})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'LangChain LangGraph difference'}`


LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployme

{'input': 'what is difference between langchain and langgraph?',
 'output': '**LangChain vs. LangGraph**\n\n| Feature | LangChain | LangGraph |\n|---------|-----------|-----------|\n| **Primary purpose** | A *building‑block* library for integrating LLMs into applications. It gives you chains, prompts, tools, and agents that you compose in code. | A *workflow engine* that lets you stitch together LangChain primitives into a **stateful, graph‑based** workflow. |\n| **Core abstraction** | **Chain** – a linear pipeline of one or more LLM calls (or other tools). | **Node / Edge** – a graph where each node can be a chain, an agent, or a custom function, and edges encode the flow between them. |\n| **State handling** | Stateless by default. You manually pass context between chains or store it yourself. | Built‑in **state** object that flows through the graph. It can be persisted, queried, and mutated by any node. |\n| **Control flow** | Linear or simple branching (via `if` statements in code)

In [31]:
agent_executor.invoke({
    "input":"What is machine learning"
})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit**Machine learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from data and improve their performance on specific tasks without being explicitly programmed for every possible scenario. In practice, ML systems identify patterns and relationships in input data, use those patterns to make predictions or decisions, and refine their models as they encounter new data. The core idea is to allow machines to generalize from past experiences to handle novel, unseen situations.

> Finished chain.


{'input': 'What is machine learning',
 'output': '**Machine learning (ML)** is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from data and improve their performance on specific tasks without being explicitly programmed for every possible scenario. In practice, ML systems identify patterns and relationships in input data, use those patterns to make predictions or decisions, and refine their models as they encounter new data. The core idea is to allow machines to generalize from past experiences to handle novel, unseen situations.'}

In [32]:
agent_executor.invoke({
    "input":"What is the paper 1706.03762 is about?"
})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c**Paper 1706.03762 – “Attention Is All You Need”**

| Item | Details |
|------|---------|
| **Authors** | Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin |
| **Published** | 2017 (ArXiv:1706.03762) – later appeared in *NeurIPS 2017* |
| **Field** | Natural Language Processing / Machine Learning |
| **Main Contribution** | Introduction of the **Transformer** architecture, a fully attention‑based model that replaces recurrent and convolutional layers in sequence‑to‑sequence tasks. |
| **Key Ideas** | 1. **Self‑attention** mechanisms (scaled dot‑product attention) that compute cont

{'input': 'What is the paper 1706.03762 is about?',
 'output': '**Paper 1706.03762 – “Attention Is All You Need”**\n\n| Item | Details |\n|------|---------|\n| **Authors** | Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N.\u202fGomez, Lukasz Kaiser, Illia Polosukhin |\n| **Published** | 2017 (ArXiv:1706.03762) – later appeared in *NeurIPS 2017* |\n| **Field** | Natural Language Processing / Machine Learning |\n| **Main Contribution** | Introduction of the **Transformer** architecture, a fully attention‑based model that replaces recurrent and convolutional layers in sequence‑to‑sequence tasks. |\n| **Key Ideas** | 1. **Self‑attention** mechanisms (scaled dot‑product attention) that compute context‑aware representations for every token in parallel.  <br>2. **Multi‑head attention** to capture information from multiple representation subspaces.  <br>3. **Positional encodings** to inject order information since the model has no recurrence.  <br>4. **Layer‑wi